# EX1. 유저 행동 패턴 분석 후 지원자 수 증가를 위한 개선점 탐색

이 노트북은 `topic1`의 raw CSV와 MySQL의 `url_summary`를 함께 사용해서 아래 흐름으로 분석합니다.

1. 유저별 이벤트 로그를 세션화한다.
2. 지원과 관련된 세션을 추린다.
3. 해당 세션의 URL을 정리하고 퍼널/경로 그룹으로 해석한다.
4. 기존 `cleaned_url_summary` 로직과 맞춰 reference summary와 대조한다.

핵심 보완점은 세션 정의입니다. `1일 간격 세션`은 해석이 쉬운 대신 하루 안의 여러 방문을 하나로 합칠 수 있으므로, 이 노트북에서는 `30분 inactivity`를 주 전략으로 두고 `1일 경계 세션`을 비교군으로 함께 둡니다.

## 분석 기준

- 주 분석 세션: 같은 유저의 이벤트 간 간격이 `30분`을 넘으면 새 세션으로 본다.
- 비교 세션: 날짜가 바뀌면 무조건 새 세션으로 나눈다.
- 지원 관련 세션: `jobs`, `api/jobs`, `search`, `apply`, `application`, `resume`, `bookmark`, `job_offer`, `notification`, `other_jobs` 중 하나 이상의 신호가 포함된 세션으로 본다.
- URL 비교 기준: raw URL을 바로 비교하지 않고, 기존 `cleaned_url_summary`와 같은 해석 규칙에 맞춰 route group / funnel stage 단위로 함께 본다.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
TOPIC1_DIR = PROJECT_ROOT / "topic1"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from analysis_utils.db import build_mysql_engine, test_mysql_connection
from analysis_utils.session_topic1 import (
    attach_session_ids,
    build_apply_session_url_summary,
    build_session_table,
    compare_apply_urls_to_reference,
    load_reference_cleaned_url_summary,
    prepare_topic1_logs,
    summarize_session_strategy,
)

sns.set_theme(style="whitegrid")
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)

# For a dry run, set ROW_LIMIT_PER_FILE to something like 300_000.
# Use None for the final full-data run.
ROW_LIMIT_PER_FILE = None

In [ ]:
# Fill MYSQL_PASSWORD in the environment before running the notebook.
MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "midproject1"),
}

engine = build_mysql_engine(**MYSQL_CONFIG)
test_mysql_connection(engine)

In [ ]:
logs_lf = prepare_topic1_logs(TOPIC1_DIR, row_limit_per_file=ROW_LIMIT_PER_FILE)

raw_log_overview = logs_lf.select(
    pl.len().alias("event_count"),
    pl.col("user_uuid").n_unique().alias("user_count"),
    pl.col("event_ts").min().alias("min_ts"),
    pl.col("event_ts").max().alias("max_ts"),
    pl.col("source_file").n_unique().alias("file_count"),
    pl.col("has_apply_signal").sum().alias("apply_signal_events"),
).collect()

raw_log_overview

In [ ]:
sessionized_30m_lf = attach_session_ids(logs_lf, gap_minutes=30, split_on_day_change=False)
sessionized_daily_lf = attach_session_ids(logs_lf, gap_minutes=24 * 60, split_on_day_change=True)

session_table_30m_lf = build_session_table(sessionized_30m_lf)
session_table_daily_lf = build_session_table(sessionized_daily_lf)

session_strategy_summary = pd.concat(
    [
        summarize_session_strategy(session_table_30m_lf, "30m_inactivity").to_pandas(),
        summarize_session_strategy(session_table_daily_lf, "1day_boundary").to_pandas(),
    ],
    ignore_index=True,
)

session_strategy_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=session_strategy_summary, x="session_strategy", y="session_count", ax=ax)
ax.set_title("Session count by sessionization strategy")
ax.set_xlabel("")
ax.set_ylabel("session_count")
plt.xticks(rotation=0)
plt.show()

세션 개수 차이가 크고, `1day_boundary`가 평균 이벤트 수를 과도하게 키운다면 하루 단위가 너무 거친 정의라는 뜻입니다. 이후 분석은 기본적으로 `30m_inactivity` 세션을 사용합니다.

In [ ]:
session_table_30m = session_table_30m_lf.collect()

apply_session_summary = (
    session_table_30m
    .filter(pl.col("has_apply_signal"))
    .select(
        pl.len().alias("apply_session_count"),
        pl.col("user_uuid").n_unique().alias("apply_user_count"),
        pl.col("event_count").mean().alias("avg_events_per_apply_session"),
        pl.col("session_duration_min").median().alias("median_apply_session_duration_min"),
    )
)

apply_session_summary

In [ ]:
apply_url_summary = build_apply_session_url_summary(sessionized_30m_lf)

apply_route_group_summary = (
    apply_url_summary.groupby("route_group", dropna=False)
    .agg(
        url_count=("url", "nunique"),
        event_count=("event_count", "sum"),
        session_count=("session_count", "sum"),
        user_count=("user_count", "sum"),
    )
    .sort_values(["event_count", "session_count"], ascending=False)
)

apply_route_group_summary.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = apply_route_group_summary.head(12).reset_index()
sns.barplot(data=plot_df, y="route_group", x="event_count", ax=ax)
ax.set_title("Top route groups inside apply-related sessions")
ax.set_xlabel("event_count")
ax.set_ylabel("route_group")
plt.show()

In [ ]:
apply_url_summary[
    [
        "url",
        "event_count",
        "session_count",
        "user_count",
        "normalized_path",
        "route_group",
        "funnel_stage",
    ]
].head(30)

In [ ]:
reference_summary = load_reference_cleaned_url_summary(engine)

comparison = compare_apply_urls_to_reference(apply_url_summary, reference_summary)
comparison["exists_in_reference"] = comparison["total_requests"].notna()

comparison[
    [
        "url",
        "event_count",
        "session_count",
        "route_group",
        "reference_route_group",
        "funnel_stage",
        "reference_funnel_stage",
        "total_requests",
        "success_rate",
        "client_error_rate",
        "server_error_rate",
        "event_share_vs_reference",
        "exists_in_reference",
    ]
].head(30)

In [ ]:
comparison_quality = pd.DataFrame(
    {
        "metric": [
            "apply_session_urls",
            "urls_matched_to_reference",
            "match_rate",
        ],
        "value": [
            len(comparison),
            int(comparison["exists_in_reference"].sum()),
            float(comparison["exists_in_reference"].mean()),
        ],
    }
)

comparison_quality

## 해석 포인트

이 노트북으로 바로 확인할 수 있는 질문은 아래와 같습니다.

- apply-related session에서는 어떤 route group이 가장 많이 등장하는가?
- job discovery -> job detail -> apply step 흐름이 실제 세션 안에서 얼마나 이어지는가?
- reference summary 대비 특정 URL/route group의 세션 내 비중이 unusually 높거나 낮은가?
- 오류율이 높은 route가 지원 관련 세션에서 반복적으로 노출되는가?

이후 확장 아이디어:

1. `Application.csv`를 합쳐 실제 지원 발생 세션과 미지원 세션을 분리
2. `JobBookmark.csv`를 합쳐 북마크가 지원 전환에 미치는 영향 확인
3. cohort 기준으로 첫 방문 후 n일 내 지원 전환율 비교
4. 세션 경로를 step sequence로 만들고 drop-off가 큰 구간 추적